In [1]:
import csv
import os
from typing import List, Dict, Any

# --- 1) Define schema fields in order ---
SCHEMA_FIELDS: List[str] = [
    "Country",
    "Year",
    "AudienceSegment",
    "Sales_USD",
    "Units_Sold",
    "Gross_Profit_USD",
    "Gross_Margin_pct",
    "Operating_Profit_USD",
    "Operating_Margin_pct",
    "Marketing_Spend_USD",
    "Price_per_Unit_USD",
    "Data_Source",
    "Confidence_Level",
    "Notes",
]

# --- 2) File path for your CSV ---
CSV_FILENAME = "berocca_sales_profit_demographics.csv"

def create_template_csv(filename: str = CSV_FILENAME) -> None:
    """
    Create a new CSV with headers only, if not already exists.
    """
    if os.path.exists(filename):
        print(f"File '{filename}' already exists. Skipping creation.")
        return
    
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=SCHEMA_FIELDS)
        writer.writeheader()
    print(f"Template CSV created at '{filename}' with headers.")

# --- 3) Helper to append a row ---
def append_row(data: Dict[str, Any], filename: str = CSV_FILENAME) -> None:
    """
    Append a row to the CSV. Missing fields are filled with empty strings.
    """
    # Validate
    missing_keys = set(SCHEMA_FIELDS) - set(data.keys())
    if missing_keys:
        # Fill missing with empty strings
        for key in missing_keys:
            data[key] = ""
    
    ordered_data = {field: data[field] for field in SCHEMA_FIELDS}
    
    with open(filename, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=SCHEMA_FIELDS)
        writer.writerow(ordered_data)
    print("Row appended:", ordered_data)

# --- 4) Basic validation functions ---
def validate_row(data: Dict[str, Any]) -> bool:
    """
    Example validations:
    - Country in {UK, US}
    - Year in [2021, 2022, 2023, 2024]
    - AudienceSegment in allowed list
    - Margins if present should be between -100 and 200
    """
    allowed_countries = {"UK", "US"}
    allowed_segments = {"Mixed", "Seniors", "Adults", "Teens"}
    allowed_years = {2021, 2022, 2023, 2024}
    
    # Country check
    if data.get("Country") not in allowed_countries:
        print("Invalid Country:", data.get("Country"))
        return False
    # Year check
    try:
        year_int = int(data.get("Year"))
        if year_int not in allowed_years:
            print("Invalid Year:", year_int)
            return False
    except Exception:
        print("Year not integer:", data.get("Year"))
        return False
    # Segment check
    if data.get("AudienceSegment") not in allowed_segments:
        print("Invalid AudienceSegment:", data.get("AudienceSegment"))
        return False
    
    # Optional margin checks
    for field in ["Gross_Margin_pct", "Operating_Margin_pct"]:
        if data.get(field) not in ("", None):
            try:
                val = float(data[field])
                if not (-100.0 <= val <= 200.0):
                    print(f"{field} out of expected bounds:", val)
                    return False
            except Exception:
                print(f"{field} not numeric:", data[field])
                return False
    
    # Passed all checks
    return True

# --- 5) Example usage ---
if __name__ == "__main__":
    create_template_csv()

    # Example placeholder row; replace with real numbers when available
    example_row = {
        "Country": "UK",
        "Year": 2024,
        "AudienceSegment": "Adults",
        "Sales_USD": "",
        "Units_Sold": "",
        "Gross_Profit_USD": "",
        "Gross_Margin_pct": "",
        "Operating_Profit_USD": "",
        "Operating_Margin_pct": "",
        "Marketing_Spend_USD": "",
        "Price_per_Unit_USD": "",
        "Data_Source": "Placeholder",
        "Confidence_Level": "Low",
        "Notes": "Awaiting purchase data"
    }

    if validate_row(example_row):
        append_row(example_row)
    else:
        print("Example row failed validation; not appended.")

Template CSV created at 'berocca_sales_profit_demographics.csv' with headers.
Row appended: {'Country': 'UK', 'Year': 2024, 'AudienceSegment': 'Adults', 'Sales_USD': '', 'Units_Sold': '', 'Gross_Profit_USD': '', 'Gross_Margin_pct': '', 'Operating_Profit_USD': '', 'Operating_Margin_pct': '', 'Marketing_Spend_USD': '', 'Price_per_Unit_USD': '', 'Data_Source': 'Placeholder', 'Confidence_Level': 'Low', 'Notes': 'Awaiting purchase data'}
